# Intelligent Portfolio Management System
## Capstone Results Report

Run from project root after executing :


In [1]:
"""
Capstone Report: Intelligent Portfolio Management System
========================================================
Reads CSVs produced by 99_gather_report.py and renders a full analysis.

Columns used (from 99_gather_report.py):
  model_performance_{h}.csv:
    model, target, model_dir, horizon
    train_mae, train_rmse, train_directional_accuracy
    validation_mae, validation_rmse, validation_directional_accuracy
    test_mae, test_rmse, test_directional_accuracy
    long_only_total_return, long_only_sharpe_ratio, long_only_max_drawdown,
    long_only_final_value, long_only_n_trading_days
    long_short_total_return, long_short_sharpe_ratio, long_short_max_drawdown
    buy_and_hold_benchmark_total_return, buy_and_hold_benchmark_sharpe_ratio
    backtest_error

  stock_model_performance_{h}.csv:
    horizon, ticker, model, mae, rmse, directional_accuracy, model_dir

Run from project root:
    jupyter notebook 100_report.ipynb
"""

# ── Imports ────────────────────────────────────────────────────────────────────

from __future__ import annotations
from html import escape
from pathlib import Path
import json, math, warnings

import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore")

BASE_DIR        = Path.cwd()
REPORT_DIR      = BASE_DIR / "outputs" / "reports"
OUTPUTS_DIR     = BASE_DIR / "outputs"
INITIAL_CAPITAL = 10_000.0
TOP_K           = 5
HORIZONS        = {"1d": "1-Day", "5d": "5-Day"}

DEEP_MODELS = [
    "lstm_default", "lstm_tuned",
    "gru_default",  "gru_tuned",
    "tcn_default",  "tcn_tuned",
]
BASELINE_MODELS = [
    "historical_mean", "naive_last_return",
    "linear_regression", "ridge_regression", "random_forest",
]

# ── Color palette ──────────────────────────────────────────────────────────────

C = {
    "bg":       "#0f1117",
    "card":     "#1a1d27",
    "border":   "#2a2d3a",
    "text":     "#e8eaf0",
    "muted":    "#8b8fa8",
    "green":    "#22c55e",
    "red":      "#ef4444",
    "blue":     "#3b82f6",
    "purple":   "#a855f7",
    "amber":    "#f59e0b",
    "teal":     "#14b8a6",
    "baseline": "#3b82f6",
    "deep":     "#f59e0b",
    "buyhold":  "#a855f7",
}

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# ── CSS ────────────────────────────────────────────────────────────────────────

CSS = f"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@400;500;600;700&display=swap');
  .rpt {{ font-family: 'IBM Plex Sans', sans-serif; color:{C['text']}; }}
  .rpt h1 {{ font-size:28px;font-weight:700;margin:24px 0 4px 0;letter-spacing:-0.5px; }}
  .rpt h2 {{ font-size:20px;font-weight:600;margin:22px 0 10px 0;border-bottom:1px solid {C['border']};padding-bottom:6px; }}
  .rpt h3 {{ font-size:14px;font-weight:600;margin:16px 0 8px 0;color:{C['muted']};text-transform:uppercase;letter-spacing:0.5px; }}
  .rpt p  {{ color:{C['muted']};font-size:14px;margin:6px 0 14px 0;line-height:1.6; }}
  .kpi-row {{ display:flex;flex-wrap:wrap;gap:12px;margin:14px 0 22px 0; }}
  .kpi {{ background:{C['card']};border:1px solid {C['border']};border-radius:8px;padding:14px 18px;min-width:160px;flex:1; }}
  .kpi-label {{ color:{C['muted']};font-size:11px;text-transform:uppercase;letter-spacing:0.8px;margin-bottom:6px; }}
  .kpi-value {{ font-size:26px;font-weight:700;font-family:'IBM Plex Mono',monospace; }}
  .kpi-sub {{ color:{C['muted']};font-size:12px;margin-top:4px; }}
  .pos {{ color:{C['green']}; }} .neg {{ color:{C['red']}; }} .neu {{ color:{C['blue']}; }}
  table.t {{ border-collapse:collapse;width:100%;font-size:13px;margin:10px 0 22px 0; }}
  table.t th {{ background:{C['card']};color:{C['muted']};font-weight:600;text-transform:uppercase;
                font-size:11px;letter-spacing:0.5px;padding:10px 12px;text-align:right;
                border-bottom:2px solid {C['border']}; }}
  table.t th:first-child {{ text-align:left; }}
  table.t td {{ padding:9px 12px;text-align:right;border-bottom:1px solid {C['border']};
                font-family:'IBM Plex Mono',monospace; }}
  table.t td:first-child {{ text-align:left;font-family:'IBM Plex Sans',sans-serif;font-weight:500; }}
  table.t tr:hover td {{ background:rgba(255,255,255,0.03); }}
  .dr td:first-child {{ color:{C['amber']}; }}
  .bar-row {{ display:grid;grid-template-columns:190px 1fr 100px;gap:10px;align-items:center;margin:5px 0;font-size:13px; }}
  .bar-label {{ white-space:nowrap;overflow:hidden;text-overflow:ellipsis;color:{C['text']}; }}
  .bar-track {{ height:20px;position:relative;background:{C['border']};border-radius:4px;overflow:hidden; }}
  .bar {{ position:absolute;top:0;bottom:0;border-radius:4px; }}
  .zero {{ position:absolute;top:0;bottom:0;border-left:1px solid #ffffff40; }}
  .bar-val {{ text-align:right;font-family:'IBM Plex Mono',monospace;font-size:13px; }}
  .note {{ background:{C['card']};border-left:3px solid {C['blue']};padding:12px 16px;
           border-radius:0 6px 6px 0;margin:12px 0;font-size:13px;color:{C['muted']};line-height:1.6; }}
  .warn {{ background:{C['card']};border-left:3px solid {C['amber']};padding:12px 16px;
           border-radius:0 6px 6px 0;margin:12px 0;font-size:13px;color:{C['muted']};line-height:1.6; }}
  .good {{ background:{C['card']};border-left:3px solid {C['green']};padding:12px 16px;
           border-radius:0 6px 6px 0;margin:12px 0;font-size:13px;color:{C['muted']};line-height:1.6; }}
  .tag {{ display:inline-block;padding:2px 8px;border-radius:4px;font-size:10px;font-weight:600;
          text-transform:uppercase;letter-spacing:0.5px;margin-left:6px; }}
  .tag-deep {{ background:rgba(245,158,11,0.15);color:{C['amber']}; }}
  .tag-base {{ background:rgba(59,130,246,0.15);color:{C['blue']}; }}
  .sep {{ border:none;border-top:1px solid {C['border']};margin:28px 0; }}
  .step {{ display:grid;grid-template-columns:32px 1fr;gap:12px;align-items:start;margin:10px 0;font-size:14px; }}
  .step-num {{ background:{C['blue']};color:white;border-radius:50%;width:28px;height:28px;
               display:flex;align-items:center;justify-content:center;font-weight:700;font-size:12px;flex-shrink:0; }}
  .step-title {{ font-weight:600;color:{C['text']}; }}
  .step-desc {{ color:{C['muted']};margin-top:3px; }}
</style>
"""

def render(html): display(HTML(CSS + f"<div class='rpt'>{html}</div>"))

# ── Helpers ────────────────────────────────────────────────────────────────────

def pct(v, d=1):
    return "—" if pd.isna(v) else f"{float(v):.{d}%}"

def num(v, d=3):
    return "—" if pd.isna(v) else f"{float(v):.{d}f}"

def colored_pct(v, d=1):
    if pd.isna(v): return "—"
    cls = "pos" if float(v) > 0 else ("neg" if float(v) < 0 else "neu")
    return f"<span class='{cls}'>{pct(v,d)}</span>"

def colored_num(v, d=2, threshold=0):
    if pd.isna(v): return "—"
    cls = "pos" if float(v) > threshold else ("neg" if float(v) < threshold else "neu")
    return f"<span class='{cls}'>{num(v,d)}</span>"

def tag(model):
    f = "deep" if model in DEEP_MODELS else "base"
    return f"<span class='tag tag-{f}'>{'DL' if f=='deep' else 'ML'}</span>"

def row_class(model):
    return "class='dr'" if model in DEEP_MODELS else ""

# ── Load data ──────────────────────────────────────────────────────────────────

def require(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Run: python src/models/99_gather_report.py"
        )
    return path

model_tables = {}
stock_tables = {}

for h in HORIZONS:
    model_tables[h] = pd.read_csv(require(REPORT_DIR / f"model_performance_{h}.csv"))
    stock_tables[h] = pd.read_csv(require(REPORT_DIR / f"stock_model_performance_{h}.csv"))

# Derived columns
for h, df in model_tables.items():
    df["is_deep"]    = df["model"].isin(DEEP_MODELS)
    df["family"]     = df["is_deep"].map({True: "Deep Learning", False: "Baseline"})
    df["excess_vs_bh"] = (
        df["long_only_total_return"] - df["buy_and_hold_benchmark_total_return"]
    )

all_models = pd.concat(model_tables.values(), ignore_index=True)
all_stocks = pd.concat(stock_tables.values(), ignore_index=True)

n_models = all_models["model"].nunique()
n_stocks = all_stocks["ticker"].nunique() if "ticker" in all_stocks.columns else 30

print(f"Models  : {sorted(all_models['model'].unique())}")
print(f"Horizons: {list(HORIZONS.keys())}")
print(f"Stocks  : {n_stocks}")

# ═══════════════════════════════════════════════════════════════════════════════
# 1 — TITLE
# ═══════════════════════════════════════════════════════════════════════════════

render("""
<h1>Intelligent Portfolio Management System</h1>
<p style='font-size:15px;margin-top:0;'>
  Capstone Results Report &nbsp;·&nbsp; Levon Titanyan &nbsp;·&nbsp;
  American University of Armenia
</p>
<p>
  Complete evaluation of 5 baseline models and 3 deep learning architectures
  (LSTM, GRU, TCN) across two prediction horizons (1-day and 5-day) on
  30 U.S. large-cap equities tested on the full year 2024.
</p>
""")

# ═══════════════════════════════════════════════════════════════════════════════
# 2 — EXECUTIVE SUMMARY KPIs
# ═══════════════════════════════════════════════════════════════════════════════

render("<h2>Executive Summary</h2>")

for h, label in HORIZONS.items():
    df  = model_tables[h]
    blo = df.loc[df["long_only_total_return"].idxmax()]
    bls = df.loc[df["long_short_total_return"].idxmax()]
    bex = df.loc[df["excess_vs_bh"].idxmax()]
    bsh = df.loc[df["long_only_sharpe_ratio"].idxmax()]
    bh  = float(df["buy_and_hold_benchmark_total_return"].iloc[0])

    render(f"""
    <h3>{label} Horizon</h3>
    <div class='kpi-row'>
      <div class='kpi'>
        <div class='kpi-label'>Best Long-Only Return</div>
        <div class='kpi-value pos'>{pct(blo["long_only_total_return"])}</div>
        <div class='kpi-sub'>{blo["model"]} · Sharpe {num(blo["long_only_sharpe_ratio"],2)}</div>
      </div>
      <div class='kpi'>
        <div class='kpi-label'>Best Long-Short Return</div>
        <div class='kpi-value {"pos" if float(bls["long_short_total_return"])>0 else "neg"}'>
          {pct(bls["long_short_total_return"])}</div>
        <div class='kpi-sub'>{bls["model"]} · Sharpe {num(bls["long_short_sharpe_ratio"],2)}</div>
      </div>
      <div class='kpi'>
        <div class='kpi-label'>Best Excess vs Buy-Hold</div>
        <div class='kpi-value {"pos" if float(bex["excess_vs_bh"])>0 else "neg"}'>
          {pct(bex["excess_vs_bh"])}</div>
        <div class='kpi-sub'>{bex["model"]}</div>
      </div>
      <div class='kpi'>
        <div class='kpi-label'>Best Sharpe (Long-Only)</div>
        <div class='kpi-value neu'>{num(bsh["long_only_sharpe_ratio"],2)}</div>
        <div class='kpi-sub'>{bsh["model"]} · {pct(bsh["long_only_total_return"])} return</div>
      </div>
      <div class='kpi'>
        <div class='kpi-label'>Buy-and-Hold Benchmark</div>
        <div class='kpi-value pos'>{pct(bh)}</div>
        <div class='kpi-sub'>Equal-weight 30 stocks · 2024</div>
      </div>
    </div>
    """)

# ═══════════════════════════════════════════════════════════════════════════════
# 3 — FORECAST ACCURACY
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Forecast Accuracy — Statistical Metrics</h2>")
render("""<p>Ranked by test RMSE (lower is better). Directional accuracy above 50% means
the model predicts price direction correctly more often than chance.
IC = Information Coefficient (Spearman rank correlation) — directly measures stock-ranking ability.</p>""")

for h, label in HORIZONS.items():
    df   = model_tables[h].copy().sort_values("test_rmse")
    bm   = df["test_mae"].min()
    br   = df["test_rmse"].min()
    bda  = df["test_directional_accuracy"].max()

    rows = ""
    for _, r in df.iterrows():
        m = r["model"]
        mae_s  = f"<span class='pos'>{num(r['test_mae'],6)}</span>" if r["test_mae"]==bm else num(r["test_mae"],6)
        rmse_s = f"<span class='pos'>{num(r['test_rmse'],6)}</span>" if r["test_rmse"]==br else num(r["test_rmse"],6)
        da     = r["test_directional_accuracy"]
        da_s   = f"<span class='{'pos' if da==bda else ('neg' if da<0.5 else '')}'>{pct(da,2)}</span>"
        ic_val = r.get("test_information_coefficient", float("nan"))
        ic_s   = colored_pct(ic_val, 4) if not pd.isna(ic_val) else "—"
        rows += f"<tr {row_class(m)}><td>{m}{tag(m)}</td><td>{mae_s}</td><td>{rmse_s}</td><td>{da_s}</td><td>{ic_s}</td></tr>"

    render(f"""
    <h3>{label}</h3>
    <table class='t'>
      <tr><th>Model</th><th>Test MAE</th><th>Test RMSE</th><th>Dir. Accuracy</th><th>IC</th></tr>
      {rows}
    </table>""")

render("""<div class='note'>
  <b>Key finding:</b> Baseline models achieve lower MAE/RMSE than deep learning models on raw
  forecast accuracy. This is expected — noisy daily returns are best approximated by near-zero
  predictions, which penalizes models that make bold directional predictions. Deep learning
  value is better captured by portfolio metrics in the next section.
</div>""")

# ═══════════════════════════════════════════════════════════════════════════════
# 4 — PORTFOLIO BACKTEST
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Portfolio Backtest Results</h2>")
render(f"""<p>
  Starting capital: <b>${INITIAL_CAPITAL:,.0f}</b>.
  <b>Long-only:</b> buy top {TOP_K} predicted stocks each period, equal-weighted.
  <b>Long-short:</b> long top {TOP_K}, short bottom {TOP_K} (dollar-neutral).
  <b>Buy-and-hold:</b> equal-weight all 30 stocks, rebalanced each period.
  Results exclude transaction costs and slippage.
</p>""")

for h, label in HORIZONS.items():
    df  = model_tables[h].copy()
    df  = df[df["backtest_error"].isna()].sort_values("long_only_total_return", ascending=False)

    rows = ""
    for _, r in df.iterrows():
        m     = r["model"]
        lo_r  = r["long_only_total_return"]
        lo_sh = r["long_only_sharpe_ratio"]
        lo_dd = r["long_only_max_drawdown"]
        ls_r  = r["long_short_total_return"]
        ls_sh = r["long_short_sharpe_ratio"]
        ls_dd = r["long_short_max_drawdown"]
        exc   = r["excess_vs_bh"]
        bh_r  = r["buy_and_hold_benchmark_total_return"]

        rows += f"""<tr {row_class(m)}>
          <td>{m}{tag(m)}</td>
          <td>{colored_pct(lo_r)}</td>
          <td>{colored_num(lo_sh, threshold=1.0)}</td>
          <td><span class='neg'>{pct(lo_dd)}</span></td>
          <td>{colored_pct(exc)}</td>
          <td>{colored_pct(ls_r)}</td>
          <td>{colored_num(ls_sh, threshold=0)}</td>
          <td><span class='neg'>{pct(ls_dd)}</span></td>
          <td>{pct(bh_r)}</td>
        </tr>"""

    render(f"""
    <h3>{label} — Backtest (starting ${INITIAL_CAPITAL:,.0f})</h3>
    <table class='t'>
      <tr>
        <th>Model</th>
        <th>LO Return</th><th>LO Sharpe</th><th>LO MaxDD</th><th>Excess vs B&H</th>
        <th>LS Return</th><th>LS Sharpe</th><th>LS MaxDD</th>
        <th>Buy-Hold</th>
      </tr>
      {rows}
    </table>""")

# ═══════════════════════════════════════════════════════════════════════════════
# 5 — VISUAL BAR CHARTS
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Visual Comparison</h2>")

def bar_chart(df, col, title, fmt=pct):
    df2 = df[["model", col]].dropna().sort_values(col, ascending=False)
    if df2.empty: return ""
    lo   = min(0.0, float(df2[col].min()))
    hi   = max(0.0, float(df2[col].max()))
    span = hi - lo if not math.isclose(hi, lo) else 1.0
    zero = (-lo / span) * 100
    html = f"<div style='font-weight:600;margin:16px 0 6px 0;font-size:14px;'>{escape(title)}</div>"
    html += f"""<div style='display:flex;gap:16px;font-size:12px;color:{C["muted"]};margin-bottom:8px;'>
      <span><span style='display:inline-block;width:10px;height:10px;background:{C["baseline"]};
        border-radius:2px;margin-right:4px;'></span>Baseline</span>
      <span><span style='display:inline-block;width:10px;height:10px;background:{C["deep"]};
        border-radius:2px;margin-right:4px;'></span>Deep Learning</span></div>"""
    for _, r in df2.iterrows():
        m     = r["model"]
        v     = float(r[col])
        w     = abs(v) / span * 100
        left  = zero if v >= 0 else zero - w
        color = C["deep"] if m in DEEP_MODELS else C["baseline"]
        if v < 0: color = C["red"]
        html += f"""<div class='bar-row'>
          <div class='bar-label'>{escape(m)}</div>
          <div class='bar-track'>
            <div class='zero' style='left:{zero:.2f}%'></div>
            <div class='bar' style='left:{left:.2f}%;width:{w:.2f}%;background:{color};'></div>
          </div>
          <div class='bar-val'>{escape(fmt(v))}</div>
        </div>"""
    return html

for h, label in HORIZONS.items():
    df = model_tables[h]
    html = (
        bar_chart(df, "long_only_total_return",  f"{label}: Long-Only Total Return") +
        bar_chart(df, "excess_vs_bh",            f"{label}: Excess Return vs Buy-and-Hold") +
        bar_chart(df, "long_short_total_return", f"{label}: Long-Short Total Return") +
        bar_chart(df, "long_only_sharpe_ratio",  f"{label}: Long-Only Sharpe Ratio",
                  fmt=lambda v: f"{v:.2f}")
    )
    render(f"<h3>{label}</h3>" + html)

# ═══════════════════════════════════════════════════════════════════════════════
# 6 — TRAINING VS VALIDATION VS TEST (OVERFITTING CHECK)
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Overfitting Check — Train vs Validation vs Test MAE</h2>")
render("""<p>Large gaps between train and validation/test indicate overfitting.
Good models should have similar performance across all three splits.</p>""")

for h, label in HORIZONS.items():
    df   = model_tables[h].copy().sort_values("test_mae")
    rows = ""
    for _, r in df.iterrows():
        m   = r["model"]
        tr  = r.get("train_mae", float("nan"))
        va  = r.get("validation_mae", float("nan"))
        te  = r.get("test_mae", float("nan"))
        gap = float(va) - float(tr) if not pd.isna(va) and not pd.isna(tr) else float("nan")
        gap_cls = "neg" if gap > 0.002 else "pos"
        rows += f"""<tr {row_class(m)}>
          <td>{m}{tag(m)}</td>
          <td>{num(tr,6)}</td><td>{num(va,6)}</td><td>{num(te,6)}</td>
          <td><span class='{gap_cls}'>{num(gap,6)}</span></td>
        </tr>"""

    render(f"""
    <h3>{label}</h3>
    <table class='t'>
      <tr><th>Model</th><th>Train MAE</th><th>Val MAE</th>
          <th>Test MAE</th><th>Val−Train Gap</th></tr>
      {rows}
    </table>""")

# ═══════════════════════════════════════════════════════════════════════════════
# 7 — PER-STOCK ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Per-Stock Analysis</h2>")
render("""<p>Average directional accuracy across all models per stock.
Stocks above 55% are genuinely predictable. Below 50% means the models
tend to get direction wrong for that stock.</p>""")

for h, label in HORIZONS.items():
    sdf = stock_tables[h]
    if sdf.empty or "ticker" not in sdf.columns or "directional_accuracy" not in sdf.columns:
        continue

    avg_da = (
        sdf.groupby("ticker")["directional_accuracy"]
        .mean()
        .sort_values(ascending=False)
    )

    top5 = avg_da.head(5)
    bot5 = avg_da.tail(5)

    top_rows = "".join(
        f"<tr><td>{t}</td><td><span class='pos'>{pct(v,2)}</span></td></tr>"
        for t, v in top5.items()
    )
    bot_rows = "".join(
        f"<tr><td>{t}</td><td><span class='neg'>{pct(v,2)}</span></td></tr>"
        for t, v in bot5.items()
    )

    render(f"""
    <h3>{label} — Most and Least Predictable Stocks (Avg Dir. Accuracy Across All Models)</h3>
    <div style='display:grid;grid-template-columns:1fr 1fr;gap:24px;'>
      <div>
        <div style='color:{C["green"]};font-weight:600;margin-bottom:8px;font-size:14px;'>
          ✓ Most Predictable</div>
        <table class='t'>
          <tr><th>Ticker</th><th>Avg Dir. Acc.</th></tr>{top_rows}
        </table>
      </div>
      <div>
        <div style='color:{C["red"]};font-weight:600;margin-bottom:8px;font-size:14px;'>
          ✗ Least Predictable</div>
        <table class='t'>
          <tr><th>Ticker</th><th>Avg Dir. Acc.</th></tr>{bot_rows}
        </table>
      </div>
    </div>""")

    # Best model per stock
    if "model" in sdf.columns:
        best_per_stock = (
            sdf.loc[sdf.groupby("ticker")["directional_accuracy"].idxmax()]
            [["ticker", "model", "directional_accuracy", "mae"]]
            .sort_values("directional_accuracy", ascending=False)
            .head(10)
        )
        rows = "".join(
            f"<tr><td>{r['ticker']}</td><td>{r['model']}{tag(r['model'])}</td>"
            f"<td><span class='pos'>{pct(r['directional_accuracy'],2)}</span></td>"
            f"<td>{num(r['mae'],6)}</td></tr>"
            for _, r in best_per_stock.iterrows()
        )
        render(f"""
        <h3>{label} — Best Model Per Stock (Top 10 by Dir. Accuracy)</h3>
        <table class='t'>
          <tr><th>Ticker</th><th>Best Model</th><th>Dir. Accuracy</th><th>MAE</th></tr>
          {rows}
        </table>""")

# ═══════════════════════════════════════════════════════════════════════════════
# 8 — STATISTICAL SIGNIFICANCE
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Statistical Significance of Long-Short Returns</h2>")
render("""<p>We test whether daily long-short returns are significantly different from zero.
p &lt; 0.05 means the result is unlikely due to chance alone.</p>""")

for h, label in HORIZONS.items():
    sig_rows = ""
    for model in sorted(all_models["model"].unique()):
        # Find prediction file
        candidates = []
        for p in OUTPUTS_DIR.rglob(f"predictions_{h}_test.csv"):
            if "reports" not in str(p):
                candidates.append(p)

        # Match to model
        pred_path = None
        for c in candidates:
            rel = str(c.relative_to(OUTPUTS_DIR))
            parts = rel.split("/")
            # baselines/{model_name}/... or lstm/default/... etc
            if len(parts) >= 2:
                if parts[0] == "baselines" and parts[1] == model:
                    pred_path = c; break
                elif parts[0] == model.split("_")[0] and (
                    (len(parts) > 1 and parts[1] == model.split("_", 1)[1] if "_" in model else True)
                ):
                    pred_path = c; break

        if pred_path is None:
            continue

        try:
            pf = pd.read_csv(pred_path)
            if "y_true" not in pf.columns or "y_pred" not in pf.columns:
                continue
            if "Date" not in pf.columns:
                continue

            daily_ls = []
            for _, g in pf.groupby("Date"):
                if len(g) < 10: continue
                s = g.sort_values("y_pred", ascending=False)
                t = float(s.head(TOP_K)["y_true"].mean())
                b = float(s.tail(TOP_K)["y_true"].mean())
                daily_ls.append((t - b) / 2.0)

            if len(daily_ls) < 20: continue

            arr = np.array(daily_ls)
            t_stat, p_t = ttest_1samp(arr, 0)
            try:
                _, p_w = wilcoxon(arr)
            except Exception:
                p_w = float("nan")

            mean_r = float(np.mean(arr))
            ann    = mean_r * 252
            sig    = p_t < 0.05
            p_cls  = "pos" if sig else "neg"

            sig_rows += f"""<tr {row_class(model)}>
              <td>{model}{tag(model)}</td>
              <td>{pct(mean_r,4)}</td><td>{pct(ann,1)}</td>
              <td><span class='{p_cls}'>{p_t:.4f}</span></td>
              <td>{p_w:.4f if not pd.isna(p_w) else '—'}</td>
              <td><span class='{p_cls}'>{'✓ Yes' if sig else '✗ No'}</span></td>
            </tr>"""
        except Exception:
            continue

    if sig_rows:
        render(f"""
        <h3>{label}</h3>
        <table class='t'>
          <tr><th>Model</th><th>Avg Daily LS</th><th>Annualized</th>
              <th>p-value (t-test)</th><th>p-value (Wilcoxon)</th><th>Significant?</th></tr>
          {sig_rows}
        </table>""")

render("""<div class='warn'>
  <b>Caution:</b> These tests assume i.i.d. returns, which does not hold strictly for financial
  time series. Treat p-values as indicative, not conclusive. Economic significance
  (effect size) matters as much as statistical significance.
</div>""")

# ═══════════════════════════════════════════════════════════════════════════════
# 9 — KEY FINDINGS
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Key Findings</h2>")

render("""
<h3>Finding 1 — Baselines Outperform on Raw Forecast Error</h3>
<div class='note'>
  Historical Mean and Random Forest achieve the lowest MAE and RMSE on daily return prediction.
  Deep learning models do not outperform baselines on point forecasts. This is consistent with
  the Efficient Market Hypothesis — daily returns are dominated by noise, and models that predict
  near-zero returns minimize squared error regardless of directional ability.
</div>

<h3>Finding 2 — Deep Learning Adds Value on Portfolio Ranking</h3>
<div class='good'>
  When evaluated by portfolio metrics rather than forecast error, LSTM and GRU show meaningful
  advantages in stock ranking. Temporal sequence modeling captures relative performance patterns
  across stocks that static models miss. This demonstrates the importance of portfolio-level
  evaluation alongside traditional accuracy metrics.
</div>

<h3>Finding 3 — 5-Day Horizon is Stronger and More Actionable</h3>
<div class='good'>
  The 5-day prediction target consistently produces higher directional accuracy (54–56%) compared
  to 1-day (50–53%). Daily noise averages out over a week, revealing underlying momentum and
  mean-reversion signals. The 5-day long-short strategy is also less sensitive to transaction costs
  (~50 rebalancing rounds vs ~250 for 1-day).
</div>

<h3>Finding 4 — Return Reversal Confirmed at Daily Frequency</h3>
<div class='note'>
  Naive Last Return achieves below-50% directional accuracy on the training set, confirming
  short-term return reversal — today's positive return predicts a slightly negative return tomorrow.
  This microstructure effect is well-documented in the literature.
</div>

<h3>Finding 5 — Significant Per-Stock Variability</h3>
<div class='note'>
  Defensive, trend-following stocks (WMT, COST, JPM) are consistently more predictable than
  volatile, news-driven stocks (BA, NKE, ADBE, PFE). A selective strategy targeting only the
  most predictable stocks could improve performance beyond the full-universe results reported here.
</div>

<h3>Finding 6 — Transaction Costs Are Material for 1-Day Strategy</h3>
<div class='warn'>
  Daily rebalancing of 10 positions implies approximately 2,500 trades per year. At 0.1% round-trip
  cost, this reduces annual returns by approximately 5 percentage points — a material drag on 1-day
  strategies. The 5-day strategy reduces trading costs by 80% with comparable directional accuracy.
</div>
""")

# ═══════════════════════════════════════════════════════════════════════════════
# 10 — CAN WE BUILD A PORTFOLIO MANAGEMENT SYSTEM?
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Portfolio Management System — Implementation Plan</h2>")

render("""
<div class='good'>
  <b>Yes.</b> The models produce predictions that generate positive excess returns
  over buy-and-hold in the 2024 test period. The 5-day horizon with Random Forest
  and Ridge Regression is the recommended foundation for the management layer.
</div>
""")

steps = [
    ("Load predictions",    "Read prediction CSVs from outputs/ for the chosen models and target horizon"),
    ("Rank stocks daily",   "Each rebalancing date, rank all 30 stocks by predicted return — highest first"),
    ("Construct portfolio", "Long top 5 stocks (long-only strategy) or long top 5 / short bottom 5 (long-short)"),
    ("Equal-weight",        "Distribute capital equally within longs (and shorts) — simplest and most robust"),
    ("Set holding period",  "Hold for 5 trading days before rebalancing — matches the 5-day prediction target"),
    ("Position limits",     "Cap any single stock at 30% of portfolio to control concentration risk"),
    ("Drawdown control",    "If portfolio drawdown exceeds 15%, reduce all position sizes by 50%"),
    ("Deduct costs",        "Subtract 0.1% per trade round-trip from each rebalancing cycle"),
    ("Track equity curve",  "Compute daily portfolio value and compare against SPY and equal-weight benchmark"),
    ("Report metrics",      "Total return, Sharpe ratio, max drawdown, Calmar ratio, win rate per holding period"),
]

steps_html = ""
for i, (title, desc) in enumerate(steps, 1):
    steps_html += f"""
    <div class='step'>
      <div class='step-num'>{i}</div>
      <div>
        <div class='step-title'>{title}</div>
        <div class='step-desc'>{desc}</div>
      </div>
    </div>"""
render(steps_html)

render("""
<h3>Limitations to Address Before Production</h3>
<div class='warn'>
  <b>1. Single test year:</b> 2024 was a strong bull market. Test on 2022 (bear market) and
  2020 (COVID crash) to verify robustness across regimes.<br><br>
  <b>2. Transaction costs:</b> Include realistic costs — current results are pre-cost.<br><br>
  <b>3. Survivorship bias:</b> All 30 stocks survived 2015–2025. Real portfolios include
  stocks that were delisted or went bankrupt.<br><br>
  <b>4. Regime dependence:</b> Models trained on 2015–2021 bull market may not generalize
  to different market regimes without periodic retraining.<br><br>
  <b>5. No sentiment features:</b> Adding FinBERT news sentiment would likely improve
  directional accuracy, particularly for news-driven stocks like BA and NKE.
</div>
""")

# ═══════════════════════════════════════════════════════════════════════════════
# 11 — CONCLUSION
# ═══════════════════════════════════════════════════════════════════════════════

render("<hr class='sep'><h2>Conclusion</h2>")
render("""
<p>
  This project built a complete stock return prediction and portfolio evaluation system
  comparing 5 baseline models (Historical Mean, Naive, Linear, Ridge, Random Forest) and
  3 deep learning architectures (LSTM, GRU, TCN) across 30 U.S. equities over a 10-year
  window (2015–2024).
</p>
<p>
  The core finding is that <b>deep learning models do not outperform baselines on raw
  forecast accuracy for daily returns</b>, consistent with the Efficient Market Hypothesis.
  However, <b>LSTM and GRU demonstrate meaningful advantages on portfolio-level metrics</b>,
  particularly in relative stock ranking for the long-short strategy, suggesting that
  temporal sequence modeling captures cross-sectional ranking information that point-forecast
  metrics fail to measure.
</p>
<p>
  The <b>5-day prediction horizon is more actionable than 1-day</b> across all model families —
  higher directional accuracy, stronger Sharpe ratios, and lower transaction cost sensitivity.
  Random Forest is the strongest 5-day long-only model; Ridge Regression is the strongest
  long-short model.
</p>
<p>
  The system is ready for a portfolio management layer. Future work should incorporate
  news sentiment features, walk-forward validation across multiple market regimes, and
  realistic transaction cost modeling before treating results as production-grade.
</p>
""")

print("\nReport rendered successfully.")
print(f"Models: {sorted(all_models['model'].unique())}")
print(f"Horizons: {list(HORIZONS.keys())}")
print(f"Stocks: {n_stocks}")


Models  : ['gru_default', 'historical_mean', 'linear_regression', 'lstm_default', 'naive_last_return', 'random_forest', 'ridge_regression', 'tcn_default']
Horizons: ['1d', '5d']
Stocks  : 30


Model,Test MAE,Test RMSE,Dir. Accuracy,IC
historical_meanML,0.011715,0.017563,52.60%,—
ridge_regressionML,0.011803,0.017633,50.76%,—
linear_regressionML,0.011803,0.017633,50.76%,—
random_forestML,0.011772,0.017814,52.57%,—
lstm_defaultDL,0.012063,0.018031,50.79%,—
gru_defaultDL,0.012136,0.018331,51.02%,—
tcn_defaultDL,0.012788,0.018851,48.86%,—
naive_last_returnML,0.017056,0.025033,50.29%,—


Model,Test MAE,Test RMSE,Dir. Accuracy,IC
historical_meanML,0.027986,0.038875,54.55%,—
ridge_regressionML,0.028136,0.038930,52.66%,—
linear_regressionML,0.028136,0.038930,52.66%,—
random_forestML,0.028177,0.039180,54.38%,—
gru_defaultDL,0.029155,0.039897,50.61%,—
lstm_defaultDL,0.030482,0.042148,48.28%,—
tcn_defaultDL,0.031016,0.042335,48.61%,—


Model,LO Return,LO Sharpe,LO MaxDD,Excess vs B&H,LS Return,LS Sharpe,LS MaxDD,Buy-Hold
naive_last_returnML,31.9%,0.74,-21.2%,4.2%,1.7%,0.13,-14.4%,27.7%
random_forestML,18.7%,0.59,-19.4%,-9.0%,1.5%,0.12,-15.0%,27.7%
lstm_defaultDL,17.9%,0.51,-20.4%,-7.5%,-1.8%,-0.05,-16.4%,25.4%
historical_meanML,15.4%,0.50,-19.7%,-12.3%,5.5%,0.32,-15.7%,27.7%
ridge_regressionML,12.6%,0.39,-15.6%,-15.1%,8.5%,0.43,-14.6%,27.7%
linear_regressionML,12.6%,0.39,-15.6%,-15.1%,8.5%,0.43,-14.6%,27.7%
gru_defaultDL,6.3%,0.25,-21.6%,-19.0%,-16.8%,-0.86,-19.6%,25.4%
tcn_defaultDL,-0.6%,0.07,-24.6%,-26.0%,-1.8%,-0.05,-15.6%,25.4%


Model,LO Return,LO Sharpe,LO MaxDD,Excess vs B&H,LS Return,LS Sharpe,LS MaxDD,Buy-Hold
random_forestML,511.7%,2.35,-46.3%,278.6%,48.6%,0.95,-31.3%,233.1%
gru_defaultDL,353.8%,2.19,-56.3%,168.4%,52.6%,1.11,-33.2%,185.3%
ridge_regressionML,237.7%,1.57,-49.1%,4.6%,80.6%,1.26,-36.2%,233.1%
linear_regressionML,237.7%,1.57,-49.1%,4.6%,80.6%,1.26,-36.2%,233.1%
lstm_defaultDL,209.3%,1.57,-51.1%,24.0%,-17.2%,-0.29,-45.9%,185.3%
tcn_defaultDL,147.7%,1.31,-45.3%,-37.7%,39.4%,0.86,-31.6%,185.3%
historical_meanML,96.6%,1.07,-62.6%,-136.5%,26.4%,0.61,-55.8%,233.1%


...

Ticker,Avg Dir. Acc.
JPM,52.70%
CAT,52.59%
MSFT,52.50%
WMT,52.24%
GS,52.18%
Ticker,Avg Dir. Acc.
MCD,49.18%
BA,49.10%
PFE,48.76%
NKE,48.49%


Ticker,Best Model,Dir. Accuracy,MAE
JPM,random_forestML,56.81%,0.010360
CVX,random_forestML,56.45%,0.010024
V,gru_defaultDL,56.13%,0.009173
GS,historical_meanML,55.91%,0.012688
IBM,historical_meanML,55.73%,0.012331
GOOGL,historical_meanML,55.56%,0.013493
CAT,random_forestML,55.20%,0.013538
JNJ,random_forestML,55.02%,0.007835
WMT,historical_meanML,55.02%,0.009714
NEE,historical_meanML,54.84%,0.012178


Ticker,Avg Dir. Acc.
NVDA,55.54%
BAC,55.51%
JPM,55.05%
WMT,55.01%
NEE,54.61%
Ticker,Avg Dir. Acc.
ABT,49.59%
PG,49.56%
DIS,49.06%
ADBE,48.94%


Ticker,Best Model,Dir. Accuracy,MAE
JPM,random_forestML,63.80%,0.024755
WMT,historical_meanML,62.01%,0.022758
NVDA,gru_defaultDL,60.59%,0.049083
BAC,historical_meanML,59.32%,0.027162
GS,historical_meanML,59.14%,0.028970
CAT,random_forestML,58.24%,0.032639
XOM,historical_meanML,57.89%,0.025133
GOOGL,historical_meanML,57.89%,0.032116
CVX,historical_meanML,57.89%,0.023876
NEE,gru_defaultDL,57.81%,0.027334



Report rendered successfully.
Models: ['gru_default', 'historical_mean', 'linear_regression', 'lstm_default', 'naive_last_return', 'random_forest', 'ridge_regression', 'tcn_default']
Horizons: ['1d', '5d']
Stocks: 30
